# Kiểm tra trực quan nhãn PKU-Market-PCB

Notebook này chỉ kiểm tra trực quan dataset `PKU-Market-PCB(Data enhanced version)`. Notebook đọc ảnh và nhãn YOLO từ `data/raw/`, vẽ bounding box cùng tên lớp, rồi ghi bản sao đã chú thích vào `results/figures/samples/`; dữ liệu gốc không bị sửa.

Mỗi lớp trong `configs/base.yaml` được lấy tối đa 10 ảnh theo phép chọn ngẫu nhiên có thể lặp lại với seed 42. Một ảnh có nhiều lớp có thể được chọn cho nhiều lớp; mỗi lần chọn vẫn được xuất thành file riêng theo lớp đích.

## Phạm vi và nguyên tắc ghép nhãn

Chỉ PKU được quét, không dùng DeepPCB hay các tác vụ bổ sung. Dataset có ảnh tăng cường trong thư mục con của `images/` nhưng nhãn tương ứng có thể nằm phẳng ở thư mục `labels/`. Vì vậy, việc ghép nhãn ưu tiên đường dẫn phản chiếu và sau đó dùng file cùng tên ở `labels/` phẳng. Ảnh không có nhãn hoặc nhãn không hợp lệ được bỏ qua và được đếm trong phần tổng kết, thay vì suy diễn nội dung nhãn.

In [1]:
from collections import Counter
from pathlib import Path
import random

import cv2
import yaml


IMAGE_SUFFIXES = {'.bmp', '.jpeg', '.jpg', '.png', '.tif', '.tiff', '.webp'}
SEED = 42
SAMPLES_PER_CLASS = 10


def project_root() -> Path:
    """Tìm thư mục chứa đồng thời data, configs và scripts."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if all((candidate / name).is_dir() for name in ('data', 'configs', 'scripts')):
            return candidate
    raise RuntimeError('Không tìm thấy thư mục gốc của dự án.')


ROOT = project_root()
DATASET_DIR = ROOT / 'data' / 'raw' / 'PKU-Market-PCB(Data enhanced version)'
CONFIG_PATH = ROOT / 'configs' / 'base.yaml'
OUTPUT_DIR = ROOT / 'results' / 'figures' / 'samples'

if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy dataset PKU: {DATASET_DIR}')
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy cấu hình lớp: {CONFIG_PATH}')

with CONFIG_PATH.open(encoding='utf-8') as config_file:
    CLASSES = yaml.safe_load(config_file)['classes']

print(f'Lớp từ {CONFIG_PATH}: {CLASSES}')
print(f'Dataset: {DATASET_DIR}')

Lớp từ /Users/Project/kltn-pcb/configs/base.yaml: ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']
Dataset: /Users/Project/kltn-pcb/data/raw/PKU-Market-PCB(Data enhanced version)


In [2]:
def label_for_image(image_path: Path, dataset_dir: Path) -> Path | None:
    """Trả về nhãn phản chiếu hoặc nhãn phẳng cùng tên của một ảnh."""
    relative = image_path.relative_to(dataset_dir)
    candidates = [image_path.with_suffix('.txt')]
    if 'images' in relative.parts:
        index = relative.parts.index('images')
        labels_dir = dataset_dir.joinpath(*relative.parts[:index], 'labels')
        mirrored = labels_dir.joinpath(*relative.parts[index + 1:]).with_suffix('.txt')
        flattened = labels_dir / image_path.with_suffix('.txt').name
        # Nhãn phẳng hỗ trợ ảnh tăng cường nằm trong thư mục con.
        candidates = [mirrored, flattened, *candidates]
    return next((path for path in candidates if path.is_file()), None)


def read_yolo_boxes(label_path: Path, class_count: int) -> list[tuple[int, float, float, float, float]]:
    """Đọc các dòng YOLO hợp lệ; lỗi định dạng dừng xử lý ảnh để tránh vẽ sai."""
    boxes = []
    for line_number, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), start=1):
        if not line.strip():
            continue
        fields = line.split()
        if len(fields) != 5:
            raise ValueError(f'{label_path}:{line_number} phải có 5 giá trị YOLO.')
        try:
            class_id = int(fields[0])
            x_center, y_center, width, height = (float(value) for value in fields[1:])
        except ValueError as error:
            raise ValueError(f'{label_path}:{line_number} chứa giá trị không hợp lệ.') from error
        if class_id not in range(class_count):
            raise ValueError(f'{label_path}:{line_number} có class_id ngoài phạm vi: {class_id}')
        if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 0 < width <= 1 and 0 < height <= 1):
            raise ValueError(f'{label_path}:{line_number} có tọa độ YOLO không hợp lệ.')
        boxes.append((class_id, x_center, y_center, width, height))
    return boxes


def draw_boxes(image, boxes, class_names):
    """Vẽ toàn bộ box hợp lệ và tên lớp lên bản sao ảnh BGR."""
    annotated = image.copy()
    height, width = annotated.shape[:2]
    for class_id, x_center, y_center, box_width, box_height in boxes:
        x1 = max(0, min(width - 1, round((x_center - box_width / 2) * width)))
        y1 = max(0, min(height - 1, round((y_center - box_height / 2) * height)))
        x2 = max(0, min(width - 1, round((x_center + box_width / 2) * width)))
        y2 = max(0, min(height - 1, round((y_center + box_height / 2) * height)))
        color = (0, 255, 0)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        label = class_names[class_id]
        (text_width, text_height), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        text_y = max(text_height + baseline, y1)
        cv2.rectangle(annotated, (x1, text_y - text_height - baseline), (x1 + text_width, text_y), color, -1)
        cv2.putText(annotated, label, (x1, text_y - baseline), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    return annotated

## Lập danh sách và lấy mẫu tái lập

Mỗi ảnh chỉ vào tập ứng viên của một lớp khi nhãn của ảnh có ít nhất một box thuộc lớp đó. Danh sách đường dẫn được sắp xếp trước khi dùng bộ sinh số ngẫu nhiên cục bộ `random.Random(42)`, nên lần chạy lại trên cùng dữ liệu cho cùng lựa chọn.

In [3]:
image_paths = sorted(
    path for path in DATASET_DIR.rglob('*')
    if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
)
candidates = {class_id: [] for class_id in range(len(CLASSES))}
skipped = Counter()

for image_path in image_paths:
    label_path = label_for_image(image_path, DATASET_DIR)
    if label_path is None:
        skipped['không có nhãn'] += 1
        continue
    try:
        boxes = read_yolo_boxes(label_path, len(CLASSES))
    except ValueError:
        skipped['nhãn không hợp lệ'] += 1
        continue
    for class_id in {box[0] for box in boxes}:
        candidates[class_id].append((image_path, label_path, boxes))

rng = random.Random(SEED)
selected = {}
for class_id, class_name in enumerate(CLASSES):
    pool = candidates[class_id]
    selected[class_id] = rng.sample(pool, k=min(SAMPLES_PER_CLASS, len(pool)))
    print(f'{class_name}: chọn {len(selected[class_id])}/{SAMPLES_PER_CLASS} từ {len(pool)} ảnh hợp lệ.')

if skipped:
    print(f'Ảnh bỏ qua: {dict(skipped)}')

missing_hole: chọn 10/10 từ 822 ảnh hợp lệ.
mouse_bite: chọn 10/10 từ 822 ảnh hợp lệ.
open_circuit: chọn 10/10 từ 826 ảnh hợp lệ.
short: chọn 10/10 từ 799 ảnh hợp lệ.
spur: chọn 10/10 từ 782 ảnh hợp lệ.
spurious_copper: chọn 10/10 từ 786 ảnh hợp lệ.


## Vẽ và xuất ảnh

Các file đầu ra được đặt tên bằng lớp đích và thứ tự mẫu, vì thế 60 lựa chọn tạo tối đa 60 file khác nhau. Nếu ảnh không đọc được hoặc không ghi được, notebook đếm lỗi và tiếp tục các mẫu còn lại. Thư mục đầu ra chỉ chứa bản sao đã chú thích; không có thao tác ghi nào vào `data/raw/`.

In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Xóa chỉ các file do notebook này tạo ở các lần chạy trước để kết quả đếm được xác định.
for old_file in OUTPUT_DIR.glob('*.jpg'):
    old_file.unlink()

written_files = []
export_errors = []
for class_id, class_name in enumerate(CLASSES):
    for sample_number, (image_path, label_path, boxes) in enumerate(selected[class_id], start=1):
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            export_errors.append(f'Không đọc được ảnh: {image_path.relative_to(DATASET_DIR)}')
            continue
        output_path = OUTPUT_DIR / f'{class_id:02d}_{class_name}_{sample_number:02d}.jpg'
        if not cv2.imwrite(str(output_path), draw_boxes(image, boxes, CLASSES)):
            export_errors.append(f'Không ghi được ảnh: {output_path}')
            continue
        written_files.append(output_path)

print(f'Đã xuất {len(written_files)} file chú thích vào {OUTPUT_DIR}.')
if export_errors:
    print('Lỗi xuất ảnh:')
    print(*export_errors, sep='\n')

expected_files = sum(len(items) for items in selected.values())
if len(written_files) != expected_files:
    raise RuntimeError(f'Chỉ xuất {len(written_files)}/{expected_files} file; xem lỗi ở trên.')
if expected_files == len(CLASSES) * SAMPLES_PER_CLASS and len(written_files) != 60:
    raise RuntimeError('Đủ ứng viên nhưng không tạo đúng 60 file.')

Đã xuất 60 file chú thích vào /Users/Project/kltn-pcb/results/figures/samples.


## Mẫu ghi nhận kiểm tra thủ công

Điền mẫu sau sau khi tự xem các file đã xuất. Mẫu không khẳng định trước tính đúng đắn, đầy đủ hay chất lượng của nhãn; chỉ ghi nhận quan sát đã được người kiểm tra xác minh.

| Lớp | File đã xem | Số box quan sát | Box và tên lớp khớp quan sát? | Nhận xét cụ thể / vị trí cần kiểm tra lại | Người kiểm tra | Ngày kiểm tra |
|---|---|---:|---|---|---|---|
| missing_hole |  |  |  |  |  |  |
| mouse_bite |  |  |  |  |  |  |
| open_circuit |  |  |  |  |  |  |
| short |  |  |  |  |  |  |
| spur |  |  |  |  |  |  |
| spurious_copper |  |  |  |  |  |  |

Nếu phát hiện vấn đề, ghi chính xác tên file đầu ra, vị trí box và mô tả quan sát. Không suy rộng kết quả từ 10 mẫu thành kết luận cho toàn bộ dataset nếu chưa có kiểm tra bổ sung.